# Process data for chromHMM multiclass classification model

## Set up wandb

In [1]:
import wandb
import anndata
import pandas as pd
import numpy as np
import os

wandb.login(host="https://api.wandb.ai")
project_name = 'human-chromhmm-fullstack'

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: avantikalal (grelu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
run = wandb.init(
    entity='grelu', project=project_name, job_type='preprocessing', name='prep',
    settings=wandb.Settings(
        program_relpath='1_data.ipynb',
        program_abspath='/code/github/gReLU-applications/chromhmm/1_data.ipynb')
)

## Load data

In [3]:
chromhmm = pd.read_table('https://public.hoffman2.idre.ucla.edu/ernst/2K9RS//full_stack/full_stack_annotation_public_release/hg38/hg38_genome_100_segments.bed.gz', header=None)
chromhmm.columns = ['chrom', 'start', 'end', 'state']
chromhmm.head()

,chrom,start,end,state
0,chr1,10000,10400,2_GapArtf2
1,chr1,10400,10600,27_Acet1
2,chr1,10600,10800,38_EnhWk4
3,chr1,10800,12800,1_GapArtf1
4,chr1,12800,13000,38_EnhWk4


## Process data

In [4]:
from grelu.data.preprocess import filter_chromosomes, filter_blacklist
from grelu.sequence.utils import resize

chromhmm = filter_chromosomes(chromhmm, include='autosomes')
chromhmm = resize(chromhmm, 1024)
chromhmm = filter_blacklist(chromhmm, 'hg38')

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Keeping 5845850 intervals
Keeping 5809104 intervals


## Get coarse-grained state labels

In [5]:
chromhmm['state'] = [
    x.split('_')[1][:-1] for x in chromhmm.state
]
chromhmm.loc[chromhmm.state.isin(['EnhA1', 'EnhA2']), 'state'] = 'EnhA'

chromhmm['state'] = chromhmm['state'].astype('category')
chromhmm.state.value_counts()  

state
Quies      1485576
Acet        639669
EnhA        613794
ReprPC      610147
Tx          561526
EnhWk       543113
HET         521161
TxWk        254518
TxEnh       190465
TxEx        121833
PromF        88429
GapArtf      51474
BivProm      48242
znf          34146
TSS          24402
DNase        20609
Name: count, dtype: int64

## Load Enformer splits

In [6]:
artifact = run.use_artifact('enformer/human_intervals:latest')
dir = artifact.download()
enformer_intervals = pd.read_table(os.path.join(dir, "data.tsv"))
enformer_intervals.head(3)

wandb:   1 of 1 files downloaded.  


,chrom,start,end,split
0,chr18,895618,1092226,train
1,chr4,113598179,113794787,train
2,chr11,18394952,18591560,train


## Split regions based on their overlap with Enformer split

In [7]:
chromhmm = chromhmm.reset_index(drop=True)
chromhmm['interval_idx'] = range(len(chromhmm))

In [8]:
import bioframe as bf
overlaps = bf.overlap(chromhmm, enformer_intervals, how='left')
overlaps.split_ = overlaps.split_.fillna('None')

overlaps = overlaps.groupby('interval_idx').split_.apply(lambda x: ''.join(list(np.unique(x))))
overlaps.value_counts()

split_
train         4963283
test           402392
valid          363619
None            76215
testtrain        1606
trainvalid       1221
testvalid         768
Name: count, dtype: int64

In [9]:
assert np.all(overlaps.index == chromhmm.interval_idx)

In [10]:
new_splits = np.array(['train'] * len(overlaps))
new_splits[[(('valid' in x) and ('train' not in x)) for x in overlaps]] = 'valid'
new_splits[[(('test' in x) and ('train' not in x) and ('valid' not in x)) for x in overlaps]] = 'test'
pd.Series(new_splits).value_counts()

train    5042325
test      402392
valid     364387
Name: count, dtype: int64

In [11]:
chromhmm['enformer_split'] = overlaps
chromhmm['split'] = new_splits

## Save dataset

In [12]:
chromhmm.to_csv('chromhmm.csv.gz', index=False) 

In [13]:
artifact = wandb.Artifact('dataset', type='dataset')
artifact.add_file(local_path='chromhmm.csv.gz', name='data.csv.gz')
run.log_artifact(artifact)

<Artifact dataset>

In [14]:
run.finish()